# M3 — the full 3D-AODL: four AODs, three degrees of freedom

**What this notebook shows.** All four channels of the 3D-AODL driven together, through the
Eq. S19 synthesizer: `TrajectorySpec` in, four RF channels out, tweezers measured against
Table I. This is the paper's core result.

| # | Physics | Prediction (paper Table I / `docs/conventions.md`) |
|---|---------|------------------------------|
| 1 | **Counter-propagating fill** — a pair diffracts only where *both* acoustic columns have arrived | dark for $t < \tau/2$, window $[D/2 - vt,\, vt - D/2]$, full at $\tau$ |
| 2 | **Pure Z** — the same $f_Z$ on all four channels is a *spherical* lens | $\bar Z = 2\,\frac{\lambda F^2}{v^2}\dot f_Z = Z(t)$, $\;X = Y = 0$, $\;\Delta F = 0$ |
| 3 | **In-plane, astigmatism-free** — the lateral term enters a pair antisymmetrically | $X = \frac{\lambda F}{v}(f_{Bx} - f_{Ax})$ with $\bar Z = \Delta F = 0$ |
| 4 | **Omnidirectional** — 3 DOF for $(X, Y, Z)$, the fourth channel spent on $\Delta F = 0$ | any short 3D path, round spot throughout |

Point 3 is the M3 headline and the direct answer to M2. Two crossed AODs cannot move a tweezer
without defocusing it (notebook 02: $\bar Z$ reached $1.9\,z_R$ *because* the array was
travelling). The counter-propagating partners `Bx`, `By` fix that: their sound runs the other
way, so their deflection **subtracts** while their lensing still **adds**
(`docs/conventions.md` §3 — $\theta_1 \propto s f$ carries the sound sign, $\theta_2 \propto \dot f$
does not). Four channels, four Table I knobs: $X$, $Y$, $\bar Z$, and $\Delta F$ held at zero.

Everything runs through the package's ordinary front door. Physics reference:
arXiv:2510.11451 (equations `S#` refer to its Supplement).

In [ ]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.special import erf, erfc

from aodl import (
    ArraySpec,
    ChannelWaveform,
    Hold,
    Lift,
    PiecewisePoly,
    ToneTrack,
    TrajectorySpec,
    Translate,
    WaveformSet,
    default_1030,
    max_z_integral,
    ramps,
    render_movie,
    simulate,
    synthesize,
)
from aodl.field.focal import spot_params
from aodl.units import MHz, um, us

P = default_1030()                 # paper hardware at lambda = 1030 nm (docs/PLAN.md 1.5)
optics = P.optics
aod = P.channels["Ax"]
tau = aod.transit_time             # acoustic transit D / v
OUT = Path("outputs")              # examples/outputs/ - gitignored


def with_order(params, order):
    "The same hardware with a different weak-drive expansion order (params.py)."
    return replace(params, channels={name: replace(a, mixing_order=order)
                                     for name, a in params.channels.items()})


P1 = with_order(P, 1)              # linear model: one tone, one beam (mixing is M2's topic)

print(f"transit time tau   = {tau / us:7.3f} us    (retardation tau/2 = {tau / 2 / us:.3f} us)")
print(f"deflection scale   = {P.deflection_scale * MHz / um:7.3f} um/MHz  (lambda F / v)")
print(f"lens scale         = {P.lens_scale:7.3e} m.s    (lambda F^2 / v^2)")
print(f"waist w0           = {optics.waist0 / um:7.3f} um,  Rayleigh range z_R = {optics.rayleigh / um:.3f} um")
print(f"Z co-chirp cost    = {1e-3 * 10 * um / (2 * P.lens_scale) / MHz:7.2f} MHz/ms per channel to hold Z = 10 um")
print(f"Eq. 1 ceiling      = {max_z_integral(P):7.3e} m.s  -> Z = 10 um for {max_z_integral(P) / (10 * um) / us:.0f} us")

## 1. A counter-propagating pair is dark until $\tau/2$

The aperture window is the whole transit-time story (`docs/conventions.md` §7). A channel with
sound direction $s$ holds drive content where $s\,u \le vt - D/2$: the half-line containing its
own transducer, bounded by the leading wavefront. `Ax` ($s = -1$) fills $u \ge D/2 - vt$ and its
partner `Bx` ($s = +1$) fills $u \le vt - D/2$ — **from opposite edges**. Light has to cross both
crystals, so what diffracts is the *intersection*

$$\big[\,D/2 - vt,\;\; vt - D/2\,\big],$$

which is **empty until $t = \tau/2$**: both wavefronts must reach a point before it can deflect
anything. Then it opens symmetrically and is the whole aperture at $t = \tau$.

That gives two clean closed forms for the diffracted power, since a constant-envelope drive
leaves the pupil a pure Gaussian with a linear phase:

$$\underbrace{\frac{P_{\rm pair}(t)}{P_\infty} = \operatorname{erf}\!\Big(\frac{\sqrt2\,h}{w_{\rm in}}\Big)}_{h \,=\, vt - D/2 \,\ge\, 0},
\qquad
\frac{P_{\rm single}(t)}{P_\infty} = \tfrac12\operatorname{erfc}\!\Big(\frac{\sqrt2\,u_{\rm edge}}{w_{\rm in}}\Big).$$

The single-AOD curve (notebook 01) is already at half power at $\tau/2$; the pair is still at
*zero* there and then rises twice as steeply. `SpotMetrics.power` integrates over the two-sided
window exactly, so the measured points sit on the curves to machine precision.

In [ ]:
def static(detuning, span):
    "One constant-detuning tone on [0, span]."
    return ChannelWaveform((ToneTrack(freq=PiecewisePoly.constant(detuning, 0.0, span)),))


span = 3 * tau
pair = WaveformSet({"Ax": static(2 * MHz, span), "Bx": static(-3 * MHz, span)}, P1)
single = WaveformSet({"Ax": static(2 * MHz, span)}, P1)

t_fill = np.linspace(0.0, 1.4 * tau, 141)
power = {name: np.array([sum(m.power for m in frame) for frame in simulate(wfs, t_fill).metrics])
         for name, wfs in (("pair (Ax + Bx)", pair), ("single (Ax)", single))}
plateau = {name: p[-1] for name, p in power.items()}

v, D = aod.sound_speed, aod.aperture
h = v * t_fill - 0.5 * D                                   # window half-width (negative = empty)
predicted = {
    "pair (Ax + Bx)": np.where(h > 0.0, erf(np.sqrt(2) * np.abs(h) / optics.w_in), 0.0),
    "single (Ax)": 0.5 * erfc(np.sqrt(2) * (-h) / optics.w_in),
}
filling = t_fill < tau                                     # past tau the model is the full line

for name, p in power.items():
    err = np.max(np.abs(p[filling] / plateau[name] - predicted[name][filling]))
    print(f"{name:16s}  max |measured - closed form| = {err:.2e} of the plateau")
    assert err < 1e-9
print(f"\npair power at 0.45 tau = {power['pair (Ax + Bx)'][t_fill < 0.45 * tau][-1]:.3e}  (strictly dark)")
print(f"pair power at 0.75 tau / plateau = {np.interp(0.75 * tau, t_fill, power['pair (Ax + Bx)']) / plateau['pair (Ax + Bx)']:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 3.6))
for name, color in (("pair (Ax + Bx)", "#3a7bd5"), ("single (Ax)", "#c1121f")):
    ax.plot(t_fill / tau, predicted[name], color="k", lw=3, alpha=0.22)
    ax.plot(t_fill / tau, power[name] / plateau[name], color=color, lw=1.6, label=name)
ax.axvline(0.5, color="k", ls="--", lw=0.9)
ax.axvline(1.0, color="k", ls="--", lw=0.9)
ax.text(0.5, 1.06, r"$\tau/2$: the wavefronts meet", ha="center", fontsize=9)
ax.text(1.0, 1.06, r"$\tau$: aperture full", ha="center", fontsize=9)
ax.set(xlabel=r"$t / \tau$", ylabel="diffracted power / plateau", ylim=(-0.03, 1.16),
       title="a pair has to fill from both sides (grey: closed forms)")
ax.legend(fontsize=8, loc="center right")
plt.tight_layout()
plt.show()

Two footnotes on that plot.

* The pair curve is **exactly zero**, not small, before $\tau/2$: `build_terms` drops every term
  when the fill window is empty, so `simulate` returns a frame with no groups at all and
  `intensity_frame` renders black. Darkness there is physics, not pruning.
* At $t = \tau$ the model switches from the window to the *full line*, because the input beam is
  modelled as an **uncropped** Gaussian (`docs/PLAN.md` decision 2): the physical $|u| \le D/2$
  crop is deliberately not applied, and the fill edge is the only aperture the field integrals
  ever see. The step that leaves is $1 - \operatorname{erf}(\sqrt2 (D/2)/w_{\rm in}) = 2\times10^{-4}$
  of the power — the light in the far wings the crystal would have clipped anyway.

## 2. Pure Z: lift the tweezer straight out of the focal plane

`Lift` changes only $Z$, and Eq. S19 buys it with the **same** co-chirp on all four channels,

$$f_Z(t) = \frac{v^2}{2\lambda F^2}\int_0^t Z\,dt' ,$$

so the lateral differences of Table I stay zero while its chirp *sum* picks up all four
channels: $\bar Z = \tfrac12\frac{\lambda F^2}{v^2}\cdot 4\dot f_Z = 2\,\text{lens\_scale}\,\dot f_Z = Z(t)$,
and $\Delta F = 0$ identically. Nothing else in the drive moves.

The move below is lift 10 µm → hold 80 µs → drop, which spends 140 µs of the 206 µs that Eq. 1
allows for a 10 µm offset (§5). Two ways of looking at the same spot are plotted: at its **own**
best-focus plane it stays diffraction-limited and round the whole way, while a camera parked at
the lab focal plane $Z = 0$ watches it swell by $\sqrt{1 + (\bar Z/z_R)^2}$ and fade.

In [ ]:
lift = TrajectorySpec(
    array=ArraySpec(1, 1),
    moves=(Lift(10 * um, 60 * us), Hold(80 * us), Lift(-10 * um, 60 * us)),
)
wfs_lift = synthesize(lift, P1)                       # band-checked (Eq. 1)
frames = np.linspace(tau, lift.duration + 2 * tau, 160)
lift_run = simulate(wfs_lift, frames)
tab = lift_run.spot_table()

x_req, y_req, z_req = lift.compile()
t_c = frames - 0.5 * tau                              # docs/conventions.md 7: no pre-compensation
z_pred = z_req(t_c)

# 1/e^2 radii in the lab focal plane Z = 0, where a fixed camera would sit
radii = np.array([np.ravel(spot_params(lift_run.terms(i), optics, 0.0)[2:])
                  for i in range(len(frames))])

print(f"peak Zbar                  = {np.max(tab['z_lab']) / um:6.3f} um = {np.max(tab['z_lab']) / optics.rayleigh:.2f} z_R")
print(f"max |Zbar - Z(t_c)|        = {np.max(np.abs(tab['z_lab'] - z_pred)) / optics.rayleigh:.2e} z_R")
print(f"max lateral drift |X|,|Y|  = {max(np.max(np.abs(tab['x'])), np.max(np.abs(tab['y']))) / optics.waist0:.2e} waists")
print(f"max |Delta F|              = {np.max(np.abs(tab['delta_f'])) / optics.rayleigh:.2e} z_R")
print(f"radii at the tracked plane = {tab['wx'].min() / optics.waist0:.6f} .. {tab['wx'].max() / optics.waist0:.6f} w0 (round: wx == wy)")
print(f"radii at Z = 0             = {radii.min() / optics.waist0:.3f} .. {radii.max() / optics.waist0:.3f} w0")
assert np.max(np.abs(tab["z_lab"] - z_pred)) < 0.02 * optics.rayleigh
assert max(np.max(np.abs(tab["x"])), np.max(np.abs(tab["y"]))) < 0.01 * optics.waist0
assert np.max(np.abs(tab["delta_f"])) < 0.02 * optics.rayleigh

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.5, 6.2), sharex=True)
(ax_z, ax_f), (ax_xy, ax_w) = axes

ax_z.plot(frames / us, z_pred / um, color="k", lw=3, alpha=0.25, label=r"requested $Z(t-\tau/2)$")
ax_z.plot(frames / us, tab["z_lab"] / um, color="#3a7bd5", lw=1.5, label=r"measured $\bar Z$")
ax_z.plot(frames / us, z_req(frames) / um, color="#c1121f", lw=1.0, ls=":", label="un-retarded $Z(t)$")
ax_z.set(ylabel="Z lab [µm]", title=r"$\bar Z$ tracks the request, half a transit late")
ax_z.legend(fontsize=8)

drive = wfs_lift.eval_table(t_c)
for name, color in (("Ax", "#3a7bd5"), ("Bx", "#8ac926"), ("Ay", "#f4a261"), ("By", "#c1121f")):
    ax_f.plot(frames / us, (P.channels[name].f_center + drive[name]["f"][0]) / MHz,
              color=color, lw=1.4, ls="-" if name.startswith("A") else "--", label=name)
ax_f.axhline(P.channels["Ax"].band[1] / MHz, color="k", lw=0.8, ls=":")
ax_f.set(ylabel="drive at the beam centre [MHz]", title="all four channels carry the same $f_Z$")
ax_f.legend(fontsize=8, ncols=2)

ax_xy.plot(frames / us, tab["x"] / optics.waist0, color="#3a7bd5", lw=1.4, label="X")
ax_xy.plot(frames / us, tab["y"] / optics.waist0, color="#f4a261", lw=1.4, ls="--", label="Y")
ax_xy.set(xlabel="t [µs]", ylabel="lateral position [waists]", ylim=(-1e-2, 1e-2),
          title="laterally static: no lateral term in the drive at all")
ax_xy.legend(fontsize=8)

ax_w.plot(frames / us, tab["wx"] / optics.waist0, color="#3a7bd5", lw=2.4, alpha=0.5,
          label=r"$w_x = w_y$ at the tracked plane")
ax_w.plot(frames / us, radii[:, 0] / optics.waist0, color="#c1121f", lw=1.4, label="$w_x$ at Z = 0")
ax_w.plot(frames / us, radii[:, 1] / optics.waist0, color="#8ac926", lw=1.4, ls="--", label="$w_y$ at Z = 0")
ax_w.set(xlabel="t [µs]", ylabel=r"1/e² radius [$w_0$]", title="round everywhere; only the depth changes")
ax_w.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3. In-plane motion with no focal shift — the M3 headline (paper Fig. 2)

`Translate` moves $X$ and $Y$ and leaves $Z$ alone. Eq. S19 splits the lateral term
**antisymmetrically inside each pair**, $-\frac{v}{2\lambda F}X$ on the `A` member and
$+\frac{v}{2\lambda F}X$ on the `B` member, so the two chirps of a pair are equal and opposite:
they *differ* in Table I's deflection and *cancel* in both of its axial sums.

$$\bar Z = \tfrac12\,\text{lens\_scale}\,(\dot f_{Ax} + \dot f_{Bx} + \dot f_{Ay} + \dot f_{By}) = 0,
\qquad \Delta F = 0 .$$

The control is the same motion done the M2 way — one tone on `Ax`, one on `Ay`, no compensation —
where the deflecting chirp *is* the lensing chirp and neither quantity can be zero while the
tweezer moves:

$$\bar Z_{\rm 2AOD} = \tfrac12\,\text{lens\_scale}\,(\dot f_{Ax} + \dot f_{Ay}), \qquad
\Delta F_{\rm 2AOD} = \text{lens\_scale}\,(\dot f_{Ax} - \dot f_{Ay}).$$

Same path in the image plane, wildly different depth and shape: this is the blue-vs-red
comparison of the paper's Fig. 2, reproduced with a $(30, 12)$ µm move in 60 µs.

In [ ]:
DX, DY, MOVE = 30 * um, 12 * um, 60 * us
translate = TrajectorySpec(array=ArraySpec(1, 1), moves=(Translate(DX, DY, MOVE),))
frames = np.linspace(tau, translate.duration + 2 * tau, 160)
t_c = frames - 0.5 * tau
scale = P.deflection_scale


def one_tone(displacement):
    "M2-style: a single tone whose deflection alone makes the move (X = -scale f_Ax)."
    return ChannelWaveform((ToneTrack(freq=ramps.min_jerk(0.0, MOVE, 0.0, -displacement / scale)),))


aodl_3d = simulate(synthesize(translate, P1), frames).spot_table()
two_aod = simulate(
    WaveformSet({"Ax": one_tone(DX), "Ay": one_tone(DY)}, P1).with_hold_until(frames[-1]),
    frames,
).spot_table()

x_req, y_req, _ = translate.compile()
for name, tab in (("3D-AODL (4 channels)", aodl_3d), ("2D-AOD (Ax + Ay)", two_aod)):
    path = max(np.max(np.abs(tab["x"] - x_req(t_c))), np.max(np.abs(tab["y"] - y_req(t_c))))
    print(f"{name:22s} path error {path / optics.waist0:8.1e} waists   "
          f"peak |Zbar| {np.max(np.abs(tab['z_lab'])) / optics.rayleigh:6.2f} z_R   "
          f"peak |sigma_astig| {np.max(np.abs(tab['sigma_astig'])):6.2f}")
assert np.max(np.abs(aodl_3d["z_lab"])) < 0.02 * optics.rayleigh
assert np.max(np.abs(aodl_3d["sigma_astig"])) < 0.02
assert np.max(np.abs(two_aod["sigma_astig"])) > 1.0

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.4, 3.6))
ax_p, ax_z, ax_s = axes

for name, tab, color in (("3D-AODL", aodl_3d, "#3a7bd5"), ("2D-AOD", two_aod, "#c1121f")):
    ax_p.plot(tab["x"] / um, tab["y"] / um, color=color, lw=2, label=name)
    ax_z.plot(frames / us, tab["z_lab"] / optics.rayleigh, color=color, lw=1.6, label=name)
    ax_s.plot(frames / us, tab["sigma_astig"], color=color, lw=1.6, label=name)
ax_p.set(xlabel="X [µm]", ylabel="Y [µm]", title="same path in the image plane")
ax_p.set_aspect("equal")
ax_p.legend(fontsize=8)
for ax, label, title in ((ax_z, r"$\bar Z / z_R$", "the 4-AOD version never leaves the plane"),
                         (ax_s, r"$\sigma_{astig} = \Delta F / z_R$", "... and never astigmatises")):
    ax.axhline(0.0, color="k", lw=0.8)
    ax.set(xlabel="t [µs]", ylabel=label, title=title)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4. Omnidirectional: an L in three dimensions

Compositions of `Translate` and `Lift` reach anywhere the band allows, so the paper's Fig. 4 "L"
becomes four lines of spec. Below: across in $x$, up, across in $y$, back down — 150 µs in total,
with the array-centre position measured from the simulated tweezer and coloured by time.

The movie is the package default view (`mode="tracked"`): the XY plane follows the scene's
best focus, **hue is the spot's own lab $Z$** (blue below the focal plane, white in it, red
above) and brightness is intensity, on one global scale. Because M3 cancels the astigmatism,
the tracked plane is a *real* focus and the spot stays sharp the whole way — the only sign of
the climb is the colour, and the XZ panel beside it. The first $\tau$ of the movie is the
startup transient of §1: black, then a fast rise as the pairs fill.

In [ ]:
l_path = TrajectorySpec(
    array=ArraySpec(1, 1),
    moves=(
        Translate(25 * um, 0.0, 40 * us),
        Lift(7 * um, 35 * us),
        Translate(0.0, 25 * um, 40 * us),
        Lift(-7 * um, 35 * us),
    ),
)
wfs_l = synthesize(l_path, P1)
frames = np.linspace(0.0, l_path.duration + 2 * tau, 100)
l_run = simulate(wfs_l, frames)
lit = np.array([bool(m) for m in l_run.metrics])          # dark frames carry no spot at all
tab = l_run.spot_table()

x_req, y_req, z_req = l_path.compile()
t_c = frames[lit] - 0.5 * tau
print(f"first lit frame at t = {frames[lit][0] / tau:.2f} tau   (pairs meet at 0.5 tau)")
print(f"path error   {max(np.max(np.abs(tab['x'] - x_req(t_c))), np.max(np.abs(tab['y'] - y_req(t_c)))) / optics.waist0:.2e} waists")
print(f"axial error  {np.max(np.abs(tab['z_lab'] - z_req(t_c))) / optics.rayleigh:.2e} z_R")
print(f"|Delta F|    {np.max(np.abs(tab['delta_f'])) / optics.rayleigh:.2e} z_R  (astigmatism-free throughout)")

fig = plt.figure(figsize=(6.4, 5.0))
ax = fig.add_subplot(projection="3d")
ax.plot(x_req(t_c) / um, y_req(t_c) / um, z_req(t_c) / um, color="k", lw=3, alpha=0.2, label="requested")
sc = ax.scatter(tab["x"] / um, tab["y"] / um, tab["z_lab"] / um, c=frames[lit] / us, cmap="viridis", s=12)
ax.set(xlabel="X [µm]", ylabel="Y [µm]", zlabel="Z lab [µm]", title="measured array centre, coloured by time")
fig.colorbar(sc, ax=ax, shrink=0.6, label="t [µs]")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import Video

movie = render_movie(l_run, OUT / "03_l_path.mp4", mode="tracked", fps=20, spectrogram_panel=True)
print(f"{movie}  ({movie.stat().st_size / 1e3:.0f} kB, {l_run.n_frames} frames)")
Video(str(movie), embed=True, html_attributes="controls loop")

## 5. The limit that M4 removes

Three degrees of freedom are free; *sustaining* them is not. Holding the array at $Z$ costs
every channel a permanent chirp $\dot f_Z = Z / (2\,\text{lens\_scale})$ — 48.5 MHz/ms for 10 µm —
so the drive walks steadily across the band (Eq. 1). Starting at the carrier, as Eq. S19 does,
the whole one-sided headroom buys

$$\Big|\int Z\,dt\Big| \le 2\,\text{lens\_scale}\,(f_{\max} - f_{\rm centre}) = 2.06\times10^{-9}\ \text{m·s}
\quad\Longrightarrow\quad Z = 10\ \mu\text{m for } 206\ \mu\text{s},$$

with the array ladder and the lateral term taking their own share of it. Ask for more and the
synthesizer refuses, with the arithmetic needed to fix it:

In [ ]:
too_long = TrajectorySpec(
    array=ArraySpec(1, 1),
    moves=(Lift(10 * um, 60 * us), Hold(400 * us), Lift(-10 * um, 60 * us)),
)
try:
    synthesize(too_long, P)
except ValueError as exc:
    print(exc)

wfs_plot = synthesize(too_long, P, check_band=False)     # documented escape hatch, plotting only
t = np.linspace(*wfs_plot.t_span, 400)
fig, ax = plt.subplots(figsize=(7.2, 3.2))
lo, hi = P.channels["Ax"].band
ax.axhspan(lo / MHz, hi / MHz, color="#8ac926", alpha=0.15)
ax.plot(t / us, (P.channels["Ax"].f_center + wfs_plot.channels["Ax"].tones[0].freq(t)) / MHz,
        color="#c1121f", lw=1.8, label=r"$f_{Ax}$ for a 400 µs hold at 10 µm")
ax.axhline(hi / MHz, color="k", lw=0.9, ls="--")
ax.set(xlabel="t [µs]", ylabel="drive [MHz]", title="Eq. 1: a sustained lift walks out of the band")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## What M4 adds

M3 gives full 3D control of an atom array with no astigmatism — but only for as long as the RF
band can hold the co-chirp. Milestone 4 removes exactly that limit with the **fading-Shepard**
waveforms of Eqs. S24–S28:

* **tone ladders instead of one chirp** — each channel carries a ladder spaced $\Delta f$; as a
  tone runs out of band it fades out while a fresh one fades in at the other edge, so $\dot f_Z$
  can be held *indefinitely* at constant total intensity;
* **shadow tweezers** — during a fade two tones are up at once, and the pupil being a product
  (Eq. S7) puts extra beams at $\pm(2\lambda F/v)\Delta f$ (Fig. S6). They interfere, which is
  why `field/focal.py` groups terms by optical frequency in the first place;
* **interlaced fading** — offsetting the $x$ and $y$ fading zones by half a period ($\xi = 1/2$)
  keeps the shadows from ever coinciding, and the Schroeder phases of Eq. S28 come back for the
  same reason they did in M2.

Notebook 04 takes the M3 machinery of this page and runs the actual product story end to end:
a 10×10 array, lifted out of the plane, traversed, and dropped — waveform file, band budget,
tracking numbers and movie.